# Введение в MapReduce модель на Python


In [2]:
from typing import Iterator
from typing import NamedTuple  # requires python 3.6+

In [3]:
def MAP(_, row: NamedTuple):
    if (row.gender == 'female'):
        yield (row.age, row)


def REDUCE(age: str, rows: Iterator[NamedTuple]):
    sum = 0
    count = 0
    for row in rows:
        sum += row.social_contacts
        count += 1
    if (count > 0):
        yield (age, sum / count)
    else:
        yield (age, 0)

Модель элемента данных

In [4]:
class User(NamedTuple):
    id: int
    age: str
    social_contacts: int
    gender: str

In [5]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [6]:
def RECORDREADER():
    return [(u.id, u) for u in input_collection]

In [7]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [8]:
def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

In [9]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output)  # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [10]:
def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

In [11]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [12]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [13]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных. 

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [14]:
def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element


def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()


def MapReduce(RECORDREADER, MAP, REDUCE):
    return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*
 
mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL 

In [15]:
from typing import NamedTuple  # requires python 3.6+
from typing import Iterator


class User(NamedTuple):
    id: int
    age: str
    social_contacts: int
    gender: str


input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]


def MAP(_, row: NamedTuple):
    if (row.gender == 'female'):
        yield (row.age, row)


def REDUCE(age: str, rows: Iterator[NamedTuple]):
    sum = 0
    count = 0
    for row in rows:
        sum += row.social_contacts
        count += 1
    if (count > 0):
        yield (age, sum / count)
    else:
        yield (age, 0)


def RECORDREADER():
    return [(u.id, u) for u in input_collection]


output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication 

In [16]:
from typing import Iterator
import numpy as np

mat = np.ones((5, 4))
vec = np.random.rand(4)  # in-memory vector in all map tasks


def MAP(coordinates: (int, int), value: int):
    i, j = coordinates
    yield (i, value * vec[j])


def REDUCE(i: int, products: Iterator[NamedTuple]):
    sum = 0
    for p in products:
        sum += p
    yield (i, sum)


def RECORDREADER():
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            yield ((i, j), mat[i, j])


output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(1.757554930517903)),
 (1, np.float64(1.757554930517903)),
 (2, np.float64(1.757554930517903)),
 (3, np.float64(1.757554930517903)),
 (4, np.float64(1.757554930517903))]

## Inverted index 

In [17]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]


def RECORDREADER():
    for (docid, document) in enumerate(documents):
        yield ("{}".format(docid), document)


def MAP(docId: str, body: str):
    for word in set(body.split(' ')):
        yield (word, docId)


def REDUCE(word: str, docIds: Iterator[str]):
    yield (word, sorted(docIds))


output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('what', ['0', '1']),
 ('it', ['0', '1', '2']),
 ('is', ['0', '1', '2']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [18]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]


def RECORDREADER():
    for (docid, document) in enumerate(documents):
        for (lineid, line) in enumerate(document.split('\n')):
            yield ("{}:{}".format(docid, lineid), line)


def MAP(docId: str, line: str):
    for word in line.split(" "):
        yield (word, 1)


def REDUCE(word: str, counts: Iterator[int]):
    sum = 0
    for c in counts:
        sum += c
    yield (word, sum)


output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [19]:
def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element


def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()


def groupbykey_distributed(map_partitions, PARTITIONER):
    global reducers
    partitions = [dict() for _ in range(reducers)]
    for map_partition in map_partitions:
        for (k2, v2) in map_partition:
            p = partitions[PARTITIONER(k2)]
            p[k2] = p.get(k2, []) + [v2]
    return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in
            enumerate(partitions)]


def PARTITIONER(obj):
    global reducers
    return hash(obj) % reducers


def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
    map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
    if COMBINER != None:
        map_partitions = map(
            lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
    reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER)  # shuffle
    reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(
        map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

    print("{} key-value pairs were sent over a network.".format(
        sum([len(vs) for (k, vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
    return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*
 
flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount 

In [20]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2


def INPUTFORMAT():
    global maps

    def RECORDREADER(split):
        for (docid, document) in enumerate(split):
            for (lineid, line) in enumerate(document.split('\n')):
                yield ("{}:{}".format(docid, lineid), line)

    split_size = int(np.ceil(len(documents) / maps))
    for i in range(0, len(documents), split_size):
        yield RECORDREADER(documents[i:i + split_size])


def MAP(docId: str, line: str):
    for word in line.split(" "):
        yield (word, 1)


def REDUCE(word: str, counts: Iterator[int]):
    sum = 0
    for c in counts:
        sum += c
    yield (word, sum)


# try to set COMBINER=REDUCER and look at the number of values sent over the network 
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2), ('banana', 2), ('it', 18), ('what', 10)]),
 (1, [('is', 18)])]

## TeraSort

In [21]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0


def INPUTFORMAT():
    global maps

    def RECORDREADER(split):
        for value in split:
            yield (value, None)

    split_size = int(np.ceil(len(input_values) / maps))
    for i in range(0, len(input_values), split_size):
        yield RECORDREADER(input_values[i:i + split_size])


def MAP(value: int, _):
    yield (value, None)


def PARTITIONER(key):
    global reducers
    global max_value
    global min_value
    bucket_size = (max_value - min_value) / reducers
    bucket_id = 0
    while ((key > (bucket_id + 1) * bucket_size) and ((bucket_id + 1) * bucket_size < max_value)):
        bucket_id += 1
    return bucket_id


def REDUCE(value: int, _):
    yield (None, value)


partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.09658502559990223)),
   (None, np.float64(0.13427512991048418)),
   (None, np.float64(0.1449604457916791)),
   (None, np.float64(0.14915409809180347)),
   (None, np.float64(0.16570433322266287)),
   (None, np.float64(0.185784710329996)),
   (None, np.float64(0.3565689621901291)),
   (None, np.float64(0.437385028002536)),
   (None, np.float64(0.4553309333407143))]),
 (1,
  [(None, np.float64(0.5144050247535249)),
   (None, np.float64(0.5495891380977078)),
   (None, np.float64(0.5553605274276263)),
   (None, np.float64(0.5840503571475837)),
   (None, np.float64(0.5856483151850672)),
   (None, np.float64(0.6309774585531457)),
   (None, np.float64(0.6330965936657401)),
   (None, np.float64(0.7464118622578841)),
   (None, np.float64(0.7621998519850806)),
   (None, np.float64(0.781735487259015)),
   (None, np.float64(0.7975664952251258)),
   (None, np.float64(0.8012640589218355)),
   (None, np.float64(0.8073347616857383)),
   (None, np.float64(0.8273239004396679))

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [22]:
numbers = [3, 7, 2, 9, 1, 8, 5]


def RECORDREADER():
    return [(i, num) for i, num in enumerate(numbers)]


def MAP(_, value):
    yield ("max", value)


def REDUCE(key, values):
    max_value = max(values)
    yield ("max", max_value)


result = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(f"Максимальное значение: {result[0][1]}")

Максимальное значение: 9


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [23]:
numbers = [10, 20, 30, 40, 60]


def RECORDREADER():
    return [(i, num) for i, num in enumerate(numbers)]


def MAP(_, value):
    yield ("avg", value)


def REDUCE(_, values):
    avg = sum(values) / len(values)
    yield ("average", avg)


result = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(f"Среднее арифметическое: {result[0][1]}")

Среднее арифметическое: 32.0


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [24]:
def groupbykey_sort(iterable):
    sorted_items = sorted(iterable, key=lambda x: x[0])

    result = []
    current_key = None
    current_values = []

    for key, value in sorted_items:
        if key != current_key:
            if current_key is not None:
                result.append((current_key, current_values))
            current_key = key
            current_values = [value]
        else:
            current_values.append(value)

    if current_key is not None:
        result.append((current_key, current_values))

    return result


test_data = [(2, 'b'), (1, 'a'), (2, 'c'), (1, 'd'), (3, 'e')]
print("Исходные данные:", test_data)
print("После groupbykey_sort:", groupbykey_sort(test_data))

Исходные данные: [(2, 'b'), (1, 'a'), (2, 'c'), (1, 'd'), (3, 'e')]
После groupbykey_sort: [(1, ['a', 'd']), (2, ['b', 'c']), (3, ['e'])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [25]:
data = [1, 2, 3, 2, 1, 4, 3, 5, 1]


def RECORDREADER():
    return [(i, val) for i, val in enumerate(data)]


def MAP(_, value):
    yield (value, None)


def REDUCE(key, _):
    yield (key, key)


result = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Уникальные значения:", [val for _, val in result])

Уникальные значения: [1, 2, 3, 4, 5]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [26]:
class Person(NamedTuple):
    id: int
    name: str
    age: int
    city: str


data = [
    Person(1, "Анна", 25, "Москва"),
    Person(2, "Иван", 30, "СПб"),
    Person(3, "Мария", 22, "Москва"),
    Person(4, "Петр", 35, "Казань")
]


def RECORDREADER():
    return [(p.id, p) for p in data]


def MAP(_, person):
    if person.city == "Москва":
        yield (person.id, person)


def REDUCE(key, values):
    for value in values:
        yield (key, value)


result = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Люди из Москвы:")
for _, person in result:
    print(f"  {person.name}, {person.age} лет")

Люди из Москвы:
  Анна, 25 лет
  Мария, 22 лет


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [27]:
class User(NamedTuple):
    id: int
    name: str
    age: int
    email: str


data = [
    User(1, "Анна", 25, "anna@mail.ru"),
    User(2, "Иван", 30, "ivan@mail.ru"),
    User(3, "Мария", 22, "maria@mail.ru")
]


def RECORDREADER():
    return [(u.id, u) for u in data]


def MAP(_, user):
    projected = (user.name, user.age)
    yield (projected, projected)


def REDUCE(key, values):
    yield (key, key)


result = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Проекция (имя, возраст):")
for (name, age), _ in result:
    print(f"  {name}: {age} лет")

Проекция (имя, возраст):
  Анна: 25 лет
  Иван: 30 лет
  Мария: 22 лет


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [28]:
set1 = [1, 2, 3]
set2 = [3, 4, 5]


def RECORDREADER():
    result = []
    for val in set1:
        result.append(("set1", val))
    for val in set2:
        result.append(("set2", val))
    return result


def MAP(source, value):
    yield (value, source)


def REDUCE(key, sources):
    yield (key, key)


result = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Объединение множеств:", [val for val, _ in sorted(result)])

Объединение множеств: [1, 2, 3, 4, 5]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [29]:
set1 = [1, 2, 3, 4]
set2 = [3, 4, 5, 6]


def RECORDREADER():
    result = []
    for val in set1:
        result.append(("A", val))
    for val in set2:
        result.append(("B", val))
    return result


def MAP(set_name, value):
    yield (value, set_name)


def REDUCE(value, sets):
    if len(set(sets)) == 2:
        yield (value, value)


result = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Пересечение множеств:", [val for val, _ in sorted(result)])

Пересечение множеств: [3, 4]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [30]:
setR = [1, 2, 3, 4]
setS = [3, 4, 5]


def RECORDREADER():
    result = []
    for val in setR:
        result.append(("R", val))
    for val in setS:
        result.append(("S", val))
    return result


def MAP(set_name, value):
    yield (value, set_name)


def REDUCE(value, sets):
    if "R" in sets and "S" not in sets:
        yield (value, value)


result = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("R - S =", [val for val, _ in sorted(result)])

R - S = [1, 2]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [31]:
class R(NamedTuple):
    a: str
    b: str


class S(NamedTuple):
    a: str
    c: str


R_data = [
    R("x", "b1"),
    R("y", "b2"),
    R("z", "b3"),
    R("x", "b4")
]

S_data = [
    S("x", "c1"),
    S("y", "c2"),
    S("w", "c3")
]


def RECORDREADER():
    result = []
    for r in R_data:
        result.append(("R", r))
    for s in S_data:
        result.append(("S", s))
    return result


def MAP(source, record):
    if source == "R":
        yield (record.a, ('R', record.b))
    else:
        yield (record.a, ('S', record.c))


def REDUCE(key, values):
    r_values = []
    s_values = []

    for val in values:
        if val[0] == 'R':
            r_values.append(val[1])
        else:  # 'S'
            s_values.append(val[1])

    for b in r_values:
        for c in s_values:
            yield (f"({key}, {b}, {c})", (key, b, c))


result = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Результат NATURAL JOIN:")
for _, triple in result:
    print(f"  {triple}")

Результат NATURAL JOIN:
  ('x', 'b1', 'c1')
  ('x', 'b4', 'c1')
  ('y', 'b2', 'c2')


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [32]:
class Sales(NamedTuple):
    product: str
    amount: int
    month: str


data = [
    Sales("яблоки", 100, "янв"),
    Sales("груши", 50, "янв"),
    Sales("яблоки", 150, "фев"),
    Sales("бананы", 200, "янв"),
    Sales("яблоки", 120, "мар"),
    Sales("груши", 70, "фев")
]


def RECORDREADER():
    return [(i, sale) for i, sale in enumerate(data)]


def MAP(_, sale):
    yield (sale.product, sale.amount)


def REDUCE(product, amounts):
    total = sum(amounts)
    yield ((product, "SUM"), total)

    maximum = max(amounts)
    yield ((product, "MAX"), maximum)

    average = sum(amounts) / len(amounts)
    yield ((product, "AVG"), average)

    count = len(amounts)
    yield ((product, "COUNT"), count)


result = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Агрегация по продуктам:")
for (product, agg_type), value in sorted(result):
    print(f"  {product}: {agg_type} = {value}")

Агрегация по продуктам:
  бананы: AVG = 200.0
  бананы: COUNT = 1
  бананы: MAX = 200
  бананы: SUM = 200
  груши: AVG = 60.0
  груши: COUNT = 2
  груши: MAX = 70
  груши: SUM = 120
  яблоки: AVG = 123.33333333333333
  яблоки: COUNT = 3
  яблоки: MAX = 150
  яблоки: SUM = 370


# 

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [33]:
matrix = np.array([[1, 2], [3, 4], [5, 6]])
vector = np.array([10, 20])


def RECORDREADER():
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            yield ((i, j), matrix[i, j])


def MAP(coords, value):
    i, j = coords
    yield (i, value * vector[j])


def REDUCE(i, products):
    result = sum(products)
    yield (i, result)


result = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Результат умножения матрицы на вектор:")
for i, val in sorted(result):
    print(f"  Строка {i}: {val}")

print(f"\nПроверка numpy: {np.dot(matrix, vector)}")

Результат умножения матрицы на вектор:
  Строка 0: 50
  Строка 1: 110
  Строка 2: 170

Проверка numpy: [ 50 110 170]


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$. 





In [34]:
# MapReduce model
def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element


def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()


def MapReduce(RECORDREADER, MAP, REDUCE):
    return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [35]:
I = 2
J = 3
K = 4 * 10
small_mat = np.random.rand(I, J)  # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J, K)


def RECORDREADER():
    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            yield ((j, k), big_mat[j, k])


def MAP(k1, v1):
    (j, k) = k1
    w = v1
    # solution code that yield(k2,v2) pairs
    for i in range(small_mat.shape[0]):
        yield ((i, k), small_mat[i, j] * w)


def REDUCE(key, values):
    (i, k) = key
    # solution code that yield(k3,v3) pairs
    result = sum(values)
    yield ((i, k), result)


Проверьте своё решение

In [36]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)


def asmatrix(reduce_output):
    reduce_output = list(reduce_output)
    I = max(i for ((i, k), vw) in reduce_output) + 1
    K = max(k for ((i, k), vw) in reduce_output) + 1
    mat = np.empty(shape=(I, K))
    for ((i, k), vw) in reduce_output:
        mat[i, k] = vw
    return mat


np.allclose(reference_solution, asmatrix(solution))  # should return true

True

In [37]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i, k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [38]:
I = 4
J = 5
K = 4 * 10
small_mat = np.random.rand(I, J)
big_mat = np.random.rand(J, K)


def RECORDREADER():
    for i in range(small_mat.shape[0]):
        for j in range(small_mat.shape[1]):
            yield (('small_mat', i, j), small_mat[i, j])

    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            yield (('big_mat', j, k), big_mat[j, k])


def MAP(key, value):
    if key[0] == 'small_mat':
        _, i, j = key
        for k in range(big_mat.shape[1]):
            yield ((i, k), ('small_mat', j, value))
    else:
        _, j, k = key
        for i in range(small_mat.shape[0]):
            yield ((i, k), ('big_mat', j, value))


def REDUCE(key, values):
    i, k = key
    m_values = {}
    n_values = {}

    for val in values:
        if val[0] == 'small_mat':
            j, m_val = val[1], val[2]
            m_values[j] = m_val
        else:  # 'N'
            j, n_val = val[1], val[2]
            n_values[j] = n_val

    result = 0
    for j in m_values:
        if j in n_values:
            result += m_values[j] * n_values[j]

    yield ((i, k), result)


result = list(MapReduce(RECORDREADER, MAP, REDUCE))

P = np.zeros((small_mat.shape[0], big_mat.shape[1]))

for (i, k), val in result:
    P[i, k] = val

print(f"\nПроверка numpy:\n{np.allclose(P, np.dot(small_mat, big_mat))}")
print("Результат умножения матриц:")
print(P)



Проверка numpy:
True
Результат умножения матриц:
[[0.77852984 1.0877646  0.78217869 1.23210543 0.81652516 1.26473962
  1.06073911 1.60913971 0.85334676 0.95805187 0.50322866 1.32408856
  1.07474179 0.82860797 0.66154586 0.82124565 0.69513475 1.00090822
  1.30176751 1.11910902 1.21512555 1.07094112 0.64606431 1.13282759
  0.46985097 1.33032513 0.95746921 1.14650012 0.94289137 0.81623641
  1.20476844 1.19568482 0.58796467 1.00350197 1.01564221 0.66540479
  0.61958595 1.24757996 1.0160697  0.96637304]
 [0.6886435  1.3721344  1.4299586  1.50283969 1.50894918 1.48725583
  1.57953685 1.90073021 1.09254837 1.67198938 1.17239825 1.1901846
  0.93642991 1.2021661  0.9409169  0.93939756 0.90864544 1.2641837
  1.35821888 1.64220855 1.34345912 1.71707445 1.11471847 1.56983996
  0.62984097 1.47525395 1.42463047 1.42781885 1.78387851 1.21764672
  1.28417469 2.02032278 1.52393721 1.44732649 1.53747206 0.78780388
  1.11815821 1.48169741 1.76840226 0.89463825]
 [1.30123891 1.71293303 1.33496561 1.30199

Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER. 

In [39]:
import numpy as np

I, J, K = 4, 5, 4 * 10
small_mat = np.random.rand(I, J)
big_mat = np.random.rand(J, K)

maps = 2
reducers = 2


def INPUTFORMAT():
    global maps

    def RECORDREADER(split):
        for elem in split:
            yield elem

    all_elements = []
    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            all_elements.append((('N', j, k), big_mat[j, k]))

    split_size = int(np.ceil(len(all_elements) / maps))
    for i in range(0, len(all_elements), split_size):
        yield RECORDREADER(all_elements[i:i + split_size])


def MAP(key, value):
    matrix_type, j, k = key
    for i in range(I):
        yield ((i, k), small_mat[i, j] * value)


def PARTITIONER(key):
    i, k = key
    return i % reducers


def REDUCE(key, values):
    i, k = key
    yield ((i, k), sum(values))


partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER)

result = np.zeros((I, K))
for part_id, part in partitioned_output:
    for (i, k), val in part:
        result[i, k] = val

print("\nПроверка numpy:")
print(np.allclose(result, np.dot(small_mat, big_mat)))
print("Результат:")
print(result)

800 key-value pairs were sent over a network.

Проверка numpy:
True
Результат:
[[0.65089273 1.75752517 0.93980906 1.11580938 1.00548849 1.02754492
  0.55986374 1.53426575 0.48111423 1.75925682 2.05368361 2.07535227
  0.68714218 1.49361057 0.72796505 0.71022275 1.14174608 1.29034531
  1.16068636 1.91333743 0.57276847 1.05302625 1.66973704 1.65354123
  1.18662613 0.63224134 1.24122851 1.3292804  1.44821202 1.43772275
  1.57978597 1.66036739 0.55514915 0.8545548  0.90641457 0.871592
  1.17855838 1.12723378 1.18675936 1.0867598 ]
 [0.4638844  0.89265882 0.7248191  0.68354407 0.39057268 0.34108835
  0.39488984 1.11662342 0.30991623 0.908688   1.20330989 1.30270898
  0.54780276 0.84904939 0.55955822 0.2989217  0.90160137 1.15191759
  0.71914773 1.1818616  0.43385872 0.83726208 1.04234135 1.17243745
  0.50967279 0.49343972 0.87454086 0.64989012 1.0235735  0.89675216
  1.04178918 0.88948104 0.53990185 0.5116205  0.65762801 0.4274373
  0.51264233 0.614183   0.59473607 0.67995649]
 [0.81887565 1

Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [40]:
I, J, K = 4, 5, 4 * 10
small_mat = np.random.rand(I, J)
big_mat = np.random.rand(J, K)

maps = 4
reducers = 2


def INPUTFORMAT():
    global maps

    def RECORDREADER(elements):
        for elem in elements:
            yield elem

    all_elements = []

    # Элементы малой матрицы с тегом 'M'
    for i in range(small_mat.shape[0]):
        for j in range(small_mat.shape[1]):
            all_elements.append((('M', i, j), small_mat[i, j]))

    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            all_elements.append((('N', j, k), big_mat[j, k]))

    np.random.shuffle(all_elements)

    split_size = int(np.ceil(len(all_elements) / maps))
    for i in range(0, len(all_elements), split_size):
        yield RECORDREADER(all_elements[i:i + split_size])


def MAP(key, value):
    matrix_type = key[0]

    if matrix_type == 'M':
        _, i, j = key
        m_val = value
        for k in range(K):
            yield ((i, k), ('M', j, m_val))

    else:
        _, j, k = key
        n_val = value
        for i in range(I):
            yield ((i, k), ('N', j, n_val))


def PARTITIONER(key):
    i, k = key
    return i % reducers


def REDUCE(key, values):
    i, k = key

    m_vals = {}
    n_vals = {}

    for val in values:
        if val[0] == 'M':
            j, m_val = val[1], val[2]
            m_vals[j] = m_val
        else:  # 'N'
            j, n_val = val[1], val[2]
            n_vals[j] = n_val

    result = 0
    for j in range(J):
        if j in m_vals and j in n_vals:
            result += m_vals[j] * n_vals[j]

    yield ((i, k), result)


print("=" * 60)
print("УМНОЖЕНИЕ МАТРИЦ - ОБЕ МАТРИЦЫ В НЕСКОЛЬКИХ RECORDREADER-АХ")
print("=" * 60)
print(f"Матрица A: {I}x{J}")
print(f"Матрица B: {J}x{K}")
print(f"Мапперов: {maps}")
print(f"Редьюсеров: {reducers}")
print("=" * 60)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER)

result = np.zeros((I, K))
for part_id, part in partitioned_output:
    print(f"\nРедьюсер {part_id} получил {len(list(part)) if hasattr(part, '__len__') else 'поток'} результатов")
    part = list(partitioned_output[part_id][1]) if isinstance(partitioned_output, list) else part
    for (i, k), val in part:
        result[i, k] = val

print("\n" + "=" * 60)
print("РЕЗУЛЬТАТ:")
print(result)

print("\n" + "=" * 60)
print("ПРОВЕРКА NUMPY:")

print({np.allclose(result, np.dot(small_mat, big_mat))})

УМНОЖЕНИЕ МАТРИЦ - ОБЕ МАТРИЦЫ В НЕСКОЛЬКИХ RECORDREADER-АХ
Матрица A: 4x5
Матрица B: 5x40
Мапперов: 4
Редьюсеров: 2
1600 key-value pairs were sent over a network.

Редьюсер 0 получил поток результатов

Редьюсер 1 получил поток результатов

РЕЗУЛЬТАТ:
[[1.46490408 1.54845915 1.0802758  1.32965809 0.68263412 1.43367457
  1.42346151 1.01890426 0.78095765 1.75909941 0.85409256 1.30105685
  1.22526538 1.07132632 1.67313016 1.50687799 1.42891442 1.22600998
  1.09414852 1.01910001 1.06212326 0.88298574 1.56008176 1.01399653
  0.64937666 1.53181996 1.15663896 1.45024176 1.18294087 1.17633145
  1.21969591 0.85575185 0.56248723 1.12274307 1.3575986  1.53238225
  1.21946996 1.40077305 1.35434769 1.36569303]
 [1.26489166 1.38700704 0.76092448 1.29657833 0.83552009 0.76267929
  1.45330614 0.95657971 0.65302686 1.72723312 1.13449803 0.8882943
  1.2050444  1.69995281 1.90766222 1.45173995 1.70652957 0.76341061
  1.23834953 1.19753913 0.8925679  1.21285015 1.75057092 0.95838811
  0.91336738 1.6365556

Случайная генерация

In [50]:
import random

I, J, K = 4, 5, 4 * 10
small_mat = np.random.rand(I, J)
big_mat = np.random.rand(J, K)

maps = 4
reducers = 2


def INPUTFORMAT_RANDOM():
    global maps

    def RECORDREADER_RANDOM():
        # Каждый RECORDREADER решает, сколько элементов отдать (от 5 до 20)
        num_elements = random.randint(5, 20)

        for _ in range(num_elements):
            if random.random() < 0.5:
                i = random.randint(0, I - 1)
                j = random.randint(0, J - 1)
                yield (('M', i, j), small_mat[i, j])
            else:
                j = random.randint(0, J - 1)
                k = random.randint(0, K - 1)
                yield (('N', j, k), big_mat[j, k])

    for _ in range(maps):
        yield RECORDREADER_RANDOM()


def MAP(key, value):
    matrix_type = key[0]

    if matrix_type == 'M':
        _, i, j = key
        m_val = value
        for k in range(K):
            yield ((i, k), ('M', j, m_val))
    else:
        _, j, k = key
        n_val = value
        for i in range(I):
            yield ((i, k), ('N', j, n_val))


def PARTITIONER(key):
    i, k = key
    return i % reducers


def REDUCE(key, values):
    i, k = key

    m_vals = {}
    n_vals = {}

    for val in values:
        if val[0] == 'M':
            j, m_val = val[1], val[2]
            m_vals[j] = m_val
        else:
            j, n_val = val[1], val[2]
            n_vals[j] = n_val

    result = 0
    for j in range(J):
        if j in m_vals and j in n_vals:
            result += m_vals[j] * n_vals[j]

    yield ((i, k), result)


partitioned_output = MapReduceDistributed(INPUTFORMAT_RANDOM, MAP, REDUCE, PARTITIONER=PARTITIONER)

result_random = np.zeros((I, K))
for part_id, part in partitioned_output:
    part_list = list(part)
    for (i, k), val in part_list:
        result_random[i, k] = val

print("\n" + "=" * 70)
print("РЕЗУЛЬТАТ СО СЛУЧАЙНЫМИ ПОДМНОЖЕСТВАМИ:")
print(result_random)

print("Сравнение с numpy: ")
print(np.allclose(result_random, np.dot(small_mat, big_mat)))

1688 key-value pairs were sent over a network.

РЕЗУЛЬТАТ СО СЛУЧАЙНЫМИ ПОДМНОЖЕСТВАМИ:
[[0.         0.07614109 0.         0.         0.11264615 0.
  0.         0.         0.         0.         0.         0.27462935
  0.5584798  0.         0.7712871  0.25377703 0.27197275 0.16540023
  0.46330409 0.         0.         0.         0.         0.
  0.21802301 0.16616937 0.         0.69949159 0.         0.07536981
  0.62729154 0.35330252 0.27153748 0.         0.01474212 0.
  0.68971212 0.         0.         0.        ]
 [0.         0.06086194 0.         0.         0.01848159 0.
  0.         0.         0.         0.         0.         0.0450578
  0.44641027 0.         0.5228359  0.13254813 0.14205178 0.02713683
  0.37033336 0.         0.         0.         0.         0.
  0.03577053 0.         0.         0.54084321 0.         0.01236575
  0.         0.         0.5254963  0.         0.01178384 0.
  0.36997207 0.         0.         0.        ]
 [0.         0.01582208 0.         0.         0.045